# Self-Supervised pre-training (SimCLR) на своих изображениях с LightlySSL

1. Берем **изображения без разметки** (папка с `.jpg/.png/...`).
2. Делаем **self-supervised предобучение** методом **SimCLR** (contrastive learning) с помощью **LightlySSL**.
3. Применяем предобученный backbone к **downstream задаче**: **поиск похожих изображений (image retrieval)**.

Почему так:
- SimCLR учит сеть выделять устойчивые визуальные признаки без меток (две аугментации одного изображения должны иметь близкие представления).
- Retrieval — очень практичная задача CV и не требует разметки, при этом хорошо демонстрирует пользу предобученных эмбеддингов.

## Что нужно от окружения
- `torch`, `torchvision`
- `lightly`
- `PIL`, `tqdm`, `matplotlib`




In [ ]:
import importlib
import sys
from pathlib import Path

def find_existing_file(candidate_paths: list[Path]) -> Path:
    for p in candidate_paths:
        if p.exists():
            return p
    raise FileNotFoundError("Could not find requirements.txt. Tried: " + ", ".join(map(str, candidate_paths)))

cwd = Path.cwd().resolve()
candidate_rel = [
    Path("requirements.txt"),
    Path("ssl_lightly/requirements.txt"),
    Path("ImageProcessingAndRecognition/ssl_lightly/requirements.txt"),
]
candidates = []
for base in [cwd, *cwd.parents]:
    candidates.extend([base / rel for rel in candidate_rel])

req_path = find_existing_file(candidates)
BASE_DIR = req_path.parent
try:
    importlib.import_module("lightly")
except ModuleNotFoundError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(req_path)])


In [ ]:
import os
import random
import time
from dataclasses import dataclass
from typing import Iterable

import matplotlib.pyplot as plt
import torch
import torchvision
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

from lightly.loss import NTXentLoss
from lightly.models.modules import SimCLRProjectionHead
from lightly.transforms.simclr_transform import SimCLRTransform

print("python:", sys.version.split()[0])
print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
import lightly
print("lightly:", lightly.__version__)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


## 1) Данные

Берем изображения из уже имеющегося датасета в репозитории:

- train: `Dataset/images/train`
- val: `Dataset/images/val`

Важно: для self-supervised **разметка не нужна**, достаточно только изображений.

Чтобы тетрадка выполнялась быстро, далее мы **берем небольшой поднабор** (например, 128–256 изображений) и учим 1–2 эпохи.
Для “настоящего” обучения увеличьте `PRETRAIN_EPOCHS`, `PRETRAIN_NUM_IMAGES`, `BATCH_SIZE`.


In [ ]:
PROJECT_ROOT = BASE_DIR.parent
TRAIN_DIR = BASE_DIR / "dataset/images/train"
VAL_DIR = BASE_DIR / "dataset/images/val"

# fallback на исходный датасет из проекта (если кто-то удалил dataset из ssl_lightly)
if not TRAIN_DIR.exists() or not VAL_DIR.exists():
    TRAIN_DIR = PROJECT_ROOT / "ImageProcessingAndRecognition/Dataset/images/train"
    VAL_DIR = PROJECT_ROOT / "ImageProcessingAndRecognition/Dataset/images/val"

assert TRAIN_DIR.exists(), f"Not found: {TRAIN_DIR}"
assert VAL_DIR.exists(), f"Not found: {VAL_DIR}"

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

def iter_image_paths(root: Path) -> Iterable[Path]:
    for dirpath, _, filenames in os.walk(root):
        for filename in filenames:
            path = Path(dirpath) / filename
            if path.suffix.lower() in IMG_EXTS:
                yield path

train_paths_all = sorted(iter_image_paths(TRAIN_DIR))
val_paths_all = sorted(iter_image_paths(VAL_DIR))
print("train images:", len(train_paths_all))
print("val images:", len(val_paths_all))


In [ ]:
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

# Попросили “обучить побольше эпох”.
# Эти параметры можно уменьшить, если нужно просто быстро проверить пайплайн.
PRETRAIN_NUM_IMAGES = len(train_paths_all)  # используем все train-изображения из dataset/
PRETRAIN_EPOCHS = 100 if device.type == "cuda" else 60
IMG_SIZE = 96
BATCH_SIZE = 32
LR = 0.2 if device.type == "cuda" else 0.15
TEMPERATURE = 0.5
NUM_WORKERS = 0
USE_AMP = (device.type == "cuda")

train_paths = train_paths_all[: min(PRETRAIN_NUM_IMAGES, len(train_paths_all))]
val_paths = val_paths_all[: min(200, len(val_paths_all))]

print("subset train:", len(train_paths))
print("subset val:", len(val_paths))


## 2) Self-Supervised pre-training: SimCLR

### Что происходит в SimCLR

Для каждого изображения делаем **две сильные аугментации** (два “view”):

- случайный crop/resize,
- color jitter,
- blur и т.п.

Сеть должна:
- сделать представления двух view **похожими** (positive pair),
- а представления разных изображений в батче — **непохожими**.

В LightlySSL это уже реализовано:
- `SimCLRTransform` — генерирует (x0, x1)
- `SimCLRProjectionHead` — MLP “голова”
- `NTXentLoss` — contrastive loss


In [ ]:
class ImagePathDataset(Dataset):
    def __init__(self, paths: list[Path], transform):
        self.paths = list(paths)
        self.transform = transform
        if not self.paths:
            raise ValueError("Empty image list")

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, idx: int):
        path = self.paths[idx]
        with Image.open(path) as img:
            img = img.convert("RGB")
        x = self.transform(img)
        return x, 0


@dataclass(frozen=True)
class BackboneSpec:
    name: str
    out_dim: int


BACKBONES = {
    "resnet18": BackboneSpec("resnet18", 512),
    "resnet34": BackboneSpec("resnet34", 512),
    "resnet50": BackboneSpec("resnet50", 2048),
}


class SimCLR(nn.Module):
    def __init__(self, backbone: nn.Module, backbone_out_dim: int):
        super().__init__()
        self.backbone = backbone
        self.projection_head = SimCLRProjectionHead(backbone_out_dim, 2048, 2048)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.backbone(x).flatten(start_dim=1)
        return self.projection_head(x)


BACKBONE_NAME = "resnet18"
spec = BACKBONES[BACKBONE_NAME]
resnet = getattr(torchvision.models, spec.name)(weights=None)
backbone = nn.Sequential(*list(resnet.children())[:-1])

model = SimCLR(backbone, spec.out_dim).to(device)
criterion = NTXentLoss(temperature=TEMPERATURE)
optimizer = torch.optim.SGD(model.parameters(), lr=LR, momentum=0.9, weight_decay=1e-4)

scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

transform = SimCLRTransform(input_size=IMG_SIZE)
train_ds = ImagePathDataset(train_paths, transform=transform)
train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
)

print("train batches:", len(train_loader))


In [ ]:
model.train()
loss_history = []
t0 = time.time()

# Baseline для сравнения: эмбеддинги до SSL-обучения (случайная инициализация backbone).
import copy
baseline_backbone_state = copy.deepcopy(model.backbone.state_dict())
baseline_model_state = copy.deepcopy(model.state_dict())

# Небольшое улучшение стабильности: cosine LR schedule по эпохам.
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=PRETRAIN_EPOCHS)

for epoch in range(1, PRETRAIN_EPOCHS + 1):
    running = 0.0
    pbar = tqdm(train_loader, desc=f"pretrain epoch {epoch}/{PRETRAIN_EPOCHS}")
    for (x0, x1), _ in pbar:
        x0 = x0.to(device, non_blocking=True)
        x1 = x1.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, enabled=USE_AMP):
            z0 = model(x0)
            z1 = model(x1)
            loss = criterion(z0, z1)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running += float(loss.detach().cpu())
        pbar.set_postfix(loss=float(loss.detach().cpu()))

    avg_loss = running / max(1, len(train_loader))
    loss_history.append(avg_loss)
    scheduler.step()
    lr_now = scheduler.get_last_lr()[0]
    print(f"epoch {epoch}: loss={avg_loss:.4f} lr={lr_now:.5f}")

print(f"pretrain done in {(time.time() - t0):.1f}s")
plt.figure(figsize=(5, 3))
plt.plot(loss_history, marker="o")
plt.title("SimCLR pretrain loss")
plt.xlabel("epoch")
plt.ylabel("NTXentLoss")
plt.grid(True)
plt.show()


In [ ]:
OUT_DIR = BASE_DIR / "output"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("output dir:", OUT_DIR)

# Быстрая проверка "стало лучше":
# Смотрим, насколько модель научилась делать два view одного и того же изображения похожими
# и насколько она отличает view разных изображений.
def pos_neg_stats(simclr_model: nn.Module, loader, num_batches: int = 2):
    simclr_model.eval()
    pos_means = []
    neg_means = []
    with torch.inference_mode():
        for i, batch in enumerate(loader):
            if i >= num_batches:
                break
            (x0, x1), _ = batch
            x0 = x0.to(device, non_blocking=True)
            x1 = x1.to(device, non_blocking=True)
            z0 = torch.nn.functional.normalize(simclr_model(x0), dim=1)
            z1 = torch.nn.functional.normalize(simclr_model(x1), dim=1)
            sim = z0 @ z1.T
            eye = torch.eye(sim.shape[0], dtype=torch.bool, device=sim.device)
            pos_means.append(sim[eye].mean().item())
            neg_means.append(sim[~eye].mean().item())
    pos = sum(pos_means) / max(1, len(pos_means))
    neg = sum(neg_means) / max(1, len(neg_means))
    return pos, neg

baseline_model = copy.deepcopy(model).to(device)
baseline_model.load_state_dict(baseline_model_state)

pos_r, neg_r = pos_neg_stats(baseline_model, train_loader)
pos_s, neg_s = pos_neg_stats(model, train_loader)

print(f"pos/neg cosine (random): pos={pos_r:.4f} neg={neg_r:.4f} margin={pos_r - neg_r:.4f}")
print(f"pos/neg cosine (ssl):    pos={pos_s:.4f} neg={neg_s:.4f} margin={pos_s - neg_s:.4f}")


## 3) Downstream: поиск похожих изображений (retrieval)

Теперь используем только **backbone** (без projection head), получаем эмбеддинги изображений и находим nearest neighbors по cosine similarity.

Почему это downstream задача:
- мы **не оптимизируем** сеть под retrieval-лосс (в этой тетрадке),
- мы просто применяем предобученные признаки к прикладной задаче.


In [ ]:
# Для retrieval используем обычный детерминированный preprocessing
val_transform = torchvision.transforms.Compose(
    [
        torchvision.transforms.Resize((IMG_SIZE, IMG_SIZE)),
        torchvision.transforms.ToTensor(),
        torchvision.transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ]
)

class SingleViewDataset(Dataset):
    def __init__(self, paths: list[Path], transform):
        self.paths = list(paths)
        self.transform = transform

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, idx: int):
        path = self.paths[idx]
        with Image.open(path) as img:
            img = img.convert("RGB")
        return self.transform(img), str(path)


val_ds = SingleViewDataset(val_paths, transform=val_transform)
val_loader = DataLoader(
    val_ds,
    batch_size=64,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
)

def compute_embeddings(embedder: nn.Module):
    embedder.eval()
    embs = []
    paths = []
    with torch.inference_mode():
        for x, p in tqdm(val_loader, desc=f"embedding ({embedder.__class__.__name__})"):
            x = x.to(device, non_blocking=True)
            e = embedder(x)
            e = torch.nn.functional.normalize(e, dim=1)
            embs.append(e.cpu())
            paths.extend(list(p))
    return torch.cat(embs, dim=0), paths

# Эмбеддинги после SSL (обученный backbone)
embedder_ssl = nn.Sequential(model.backbone, nn.Flatten(start_dim=1)).to(device)
embs_ssl, paths = compute_embeddings(embedder_ssl)

# Baseline: эмбеддинги на backbone ДО обучения (случайная инициализация)
import copy
baseline_backbone = copy.deepcopy(model.backbone).to(device)
baseline_backbone.load_state_dict(baseline_backbone_state)
embedder_random = nn.Sequential(baseline_backbone, nn.Flatten(start_dim=1)).to(device)
embs_random, _ = compute_embeddings(embedder_random)

print("embeddings (ssl):", tuple(embs_ssl.shape))
print("embeddings (random):", tuple(embs_random.shape))


In [ ]:
def compute_rows(embs: torch.Tensor, query_indices: list[int], topk: int):
    sims = embs @ embs.T
    rows = []
    for qi in query_indices:
        vals, idxs = torch.topk(sims[qi], k=topk, largest=True)
        rows.append((qi, idxs.tolist(), vals.tolist()))
    return rows

n = embs_ssl.shape[0]
TOPK = min(6, n)
NUM_QUERIES = min(5, n)

# Фиксируем одни и те же query для сравнения "random" vs "ssl"
gen = torch.Generator().manual_seed(SEED)
query_indices = torch.randperm(n, generator=gen)[:NUM_QUERIES].tolist()

rows_ssl = compute_rows(embs_ssl, query_indices, TOPK)
rows_random = compute_rows(embs_random, query_indices, TOPK)

def offdiag_mean(embs: torch.Tensor) -> float:
    sims = embs @ embs.T
    n = sims.shape[0]
    if n < 2:
        return 0.0
    return float((sims.sum() - n) / (n * (n - 1)))

print("offdiag cosine (random backbone):", round(offdiag_mean(embs_random), 4))
print("offdiag cosine (ssl backbone):", round(offdiag_mean(embs_ssl), 4))

print("example query:", query_indices[0])
print("topk paths (random):")
for idx in rows_random[0][1]:
    print("-", paths[idx])
print("topk paths (ssl):")
for idx in rows_ssl[0][1]:
    print("-", paths[idx])


In [ ]:
def plot_retrieval(rows, title: str, out_png: Path):
    fig_w = max(8, 2 * TOPK)
    fig_h = max(2, 2 * len(rows))
    fig, axes = plt.subplots(len(rows), TOPK, figsize=(fig_w, fig_h))
    if len(rows) == 1:
        axes = [axes]

    for r, (_, idxs, vals) in enumerate(rows):
        for c, (idx, sim) in enumerate(zip(idxs, vals)):
            ax = axes[r][c]
            with Image.open(paths[idx]) as img:
                ax.imshow(img.convert("RGB"))
            cell_title = "query" if c == 0 else f"{sim:.2f}"
            ax.set_title(cell_title, fontsize=10)
            ax.axis("off")

    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(out_png, dpi=160)
    print("saved:", out_png)
    plt.show()

plot_retrieval(rows_random, "Retrieval (random init)", OUT_DIR / "retrieval_grid_random.png")
plot_retrieval(rows_ssl, "Retrieval (after SimCLR)", OUT_DIR / "retrieval_grid_ssl.png")
